# TTM Zero-Shot Error Metric Analysis

Computes **MSE, RMSE, MAE, MASE, MAPE, sMAPE** from the saved outputs in `ttm_results_v1/`.

- Predictions and actuals are **inverse-scaled** before computing metrics.
- For each site × pollutant the error is the **mean over the 12 prediction steps**, then averaged across all rolling windows.
- MASE uses a naïve one-step-ahead baseline computed on the context (past) values.
- Final output: one row per *site × pollutant* with all six metrics.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import pickle as pkl
from tqdm import tqdm

RESULTS_DIR = "/home/student/rishi/ttm_results_v1"
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 12

## Helper Functions — Inverse Scaling & Error Metrics

In [ ]:
def inverse_scale(arr: np.ndarray, mean: np.ndarray, scale: np.ndarray) -> np.ndarray:
    """Inverse standard-scaling: x_orig = x_scaled * scale + mean.
    
    Args:
        arr: (..., n_channels) scaled values
        mean: (n_channels,) per-channel means
        scale: (n_channels,) per-channel stds
    Returns:
        Inverse-scaled array with same shape as `arr`.
    """
    return arr * scale + mean


# ── Per-pollutant metric functions ──────────────────────────────────────
# All functions accept (actual, pred) with shapes (N, horizon) and return
# a scalar: the metric averaged over all N windows and `horizon` steps.

def mse(actual: np.ndarray, pred: np.ndarray) -> float:
    return float(np.mean((actual - pred) ** 2))

def rmse(actual: np.ndarray, pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((actual - pred) ** 2)))

def mae(actual: np.ndarray, pred: np.ndarray) -> float:
    return float(np.mean(np.abs(actual - pred)))

def mape(actual: np.ndarray, pred: np.ndarray, eps: float = 1e-8) -> float:
    """Mean Absolute Percentage Error (%).  Values where |actual| < eps are excluded."""
    mask = np.abs(actual) > eps
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100)

def smape(actual: np.ndarray, pred: np.ndarray, eps: float = 1e-8) -> float:
    """Symmetric MAPE (%).  Denominator clipped to eps to avoid division by zero."""
    denom = (np.abs(actual) + np.abs(pred)) / 2.0
    denom = np.clip(denom, eps, None)
    return float(np.mean(np.abs(actual - pred) / denom) * 100)

def mase(actual: np.ndarray, pred: np.ndarray, past: np.ndarray) -> float:
    """Mean Absolute Scaled Error.
    
    Naive baseline = one-step-ahead forecast on past (context) values:
        MAE_naive = mean(|past[t] - past[t-1]|)  for t = 1..T-1
    MASE = MAE(actual, pred) / MAE_naive
    
    Args:
        actual: (N, horizon) inverse-scaled ground truth
        pred:   (N, horizon) inverse-scaled predictions
        past:   (N, context_len) inverse-scaled past values for same pollutant
    """
    # Naive MAE on the context windows  (N, context_len-1)
    naive_errors = np.abs(np.diff(past, axis=1))
    mae_naive = np.mean(naive_errors)
    if mae_naive < 1e-12:
        return np.nan
    return float(np.mean(np.abs(actual - pred)) / mae_naive)

## Compute Metrics for Every Site

Iterates over all site folders, loads the three artefacts, inverse-scales, and computes metrics per pollutant.

In [ ]:
rows = []

site_dirs = sorted(
    d for d in os.listdir(RESULTS_DIR)
    if os.path.isdir(os.path.join(RESULTS_DIR, d))
)

for site_name in tqdm(site_dirs, desc="Computing metrics"):
    site_path = os.path.join(RESULTS_DIR, site_name)

    # ── Load artefacts ──────────────────────────────────────────────
    dataset = torch.load(os.path.join(site_path, "dataset.pt"), weights_only=False)
    preds_tensor = torch.load(os.path.join(site_path, "predictions.pt"), weights_only=False)
    with open(os.path.join(site_path, "scaler_params.pkl"), "rb") as f:
        scaler = pkl.load(f)

    future_vals = dataset["future_values"].numpy()   # (N, 12, 6)
    past_vals   = dataset["past_values"].numpy()      # (N, 512, 6)
    preds_np    = preds_tensor.numpy()                 # (N, 12, 6)

    mean_ = np.array(scaler["mean_"])                  # (6,)
    scale_ = np.array(scaler["scale_"])                # (6,)
    target_columns = scaler["target_columns"]          # list of 6

    # ── Inverse-scale everything ────────────────────────────────────
    future_inv = inverse_scale(future_vals, mean_, scale_)   # (N, 12, 6)
    preds_inv  = inverse_scale(preds_np, mean_, scale_)      # (N, 12, 6)
    past_inv   = inverse_scale(past_vals, mean_, scale_)     # (N, 512, 6)

    # ── Metrics per pollutant ───────────────────────────────────────
    for col_idx, col_name in enumerate(target_columns):
        a = future_inv[:, :, col_idx]   # (N, 12)
        p = preds_inv[:, :, col_idx]    # (N, 12)
        c = past_inv[:, :, col_idx]     # (N, 512)

        rows.append({
            "site":              site_name,
            "context_length":    CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "pollutant":         col_name,
            "n_windows":         a.shape[0],
            "MSE":               mse(a, p),
            "RMSE":              rmse(a, p),
            "MAE":               mae(a, p),
            "MASE":              mase(a, p, c),
            "MAPE":              mape(a, p),
            "sMAPE":             smape(a, p),
        })

metrics_df = pd.DataFrame(rows)
print(f"Shape: {metrics_df.shape}")
metrics_df.head(12)

## Summary Statistics by Pollutant

Average metrics across all sites for each pollutant.

In [ ]:
metric_cols = ["MSE", "RMSE", "MAE", "MASE", "MAPE", "sMAPE"]
summary = metrics_df.groupby("pollutant")[metric_cols].agg(["mean", "median", "std"]).round(4)
summary

## Export

In [ ]:
out_path = "/home/student/rishi/ttm_zeroshot_metrics.csv"
metrics_df.to_csv(out_path, index=False)
print(f"Saved {len(metrics_df)} rows → {out_path}")